# NormBank Cleaning

This notebook inspects the NormBank raw file, visualizes label/split balance,
standardizes the schema, and prepares the processed outputs.


In [ ]:
from __future__ import annotations

from pathlib import Path
from ast import literal_eval
import csv
import hashlib
import json
import re
import shutil

import matplotlib.pyplot as plt
import seaborn as sns

try:
    import pandas as pd
except Exception as exc:
    raise RuntimeError("pandas is required to use this cleaning notebook.") from exc

from IPython.display import display


try:
    import pyarrow.parquet as pq
except Exception:
    pq = None


sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

def find_project_root() -> Path:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "Data").exists() or (candidate / "data").exists():
            return candidate
    return Path.cwd().resolve()


ROOT = find_project_root()
DATA_ROOT = ROOT / "Data" if (ROOT / "Data").exists() else ROOT / "data"
RAW_DIR = DATA_ROOT / "raw/normbank"
OUT_DIR = DATA_ROOT / "processed" / "normbank"
SAVE_OUTPUTS = False

print("Project root:", ROOT)
print("Raw dir:", RAW_DIR)
print("Output dir:", OUT_DIR)
print("SAVE_OUTPUTS:", SAVE_OUTPUTS)


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def normalize_for_csv(value):
    if value is None:
        return ""
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return value


def write_jsonl(path: Path, rows) -> None:
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def write_csv(path: Path, rows, fieldnames=None) -> None:
    ensure_dir(path.parent)
    if not rows:
        return
    if fieldnames is None:
        fieldnames = []
        seen = set()
        for row in rows:
            for key in row:
                if key not in seen:
                    seen.add(key)
                    fieldnames.append(key)
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({k: normalize_for_csv(row.get(k)) for k in fieldnames})


def plot_count(series, title: str, top_n: int = 15):
    counts = series.fillna("<missing>").astype(str).value_counts().head(top_n)
    if counts.empty:
        print(f"No values available for {title}")
        return
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(x=counts.index, y=counts.values, ax=ax, color="#4C72B0")
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("count")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()


def plot_text_length(series, title: str):
    lengths = series.fillna("").astype(str).str.len()
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.histplot(lengths, bins=30, ax=ax, color="#55A868")
    ax.set_title(title)
    ax.set_xlabel("characters")
    plt.tight_layout()
    plt.show()


In [ ]:
parquet_files = sorted(RAW_DIR.glob("*.parquet"))
csv_files = sorted(RAW_DIR.glob("*.csv"))

if parquet_files:
    if pq is None:
        raise RuntimeError("pyarrow is required to read parquet files.")
    raw_df = pd.concat([pq.read_table(path).to_pandas() for path in parquet_files], ignore_index=True)
    raw_df["source_file"] = raw_df.get("split", "parquet")
    raw_df["split"] = raw_df.get("split", "")
elif csv_files:
    raw_df = pd.concat([pd.read_csv(path).assign(source_file=path.name) for path in csv_files], ignore_index=True)
else:
    raise FileNotFoundError(f"No CSV or parquet files found under {RAW_DIR}")

print("Raw shape:", raw_df.shape)
display(raw_df.head())
display(pd.DataFrame({"column": raw_df.columns, "missing": raw_df.isna().sum().values}).sort_values("missing", ascending=False))


In [ ]:
text_series = raw_df.get("norm", raw_df.get("setting-behavior", pd.Series(dtype=str))).fillna("").astype(str).str.strip()
label_series = raw_df.get("label", pd.Series(dtype=str)).fillna("").astype(str).str.strip()
split_series = raw_df.get("split", pd.Series(dtype=str)).fillna("").astype(str).str.strip()

plot_count(label_series[label_series != ""], "NormBank label distribution")
plot_count(split_series.replace("", "<blank>"), "NormBank split distribution")
plot_text_length(text_series, "NormBank text length distribution")

display(pd.DataFrame({
    "blank_text": [int((text_series == "").sum())],
    "blank_label": [int((label_series == "").sum())],
    "blank_split": [int((split_series == "").sum())],
}))


In [ ]:
cleaned_df = pd.DataFrame({
    "text": raw_df.get("norm", raw_df.get("setting-behavior", "")).fillna("").astype(str).str.strip(),
    "label": raw_df.get("label", "").fillna("").astype(str).str.strip(),
    "dataset": "normbank",
    "task": "norm_classification",
    "split": raw_df.get("split", "").fillna("").astype(str).str.strip(),
    "source_file": raw_df["source_file"].astype(str),
})
metadata_cols = [c for c in raw_df.columns if c not in {"norm", "setting-behavior", "label", "split", "source_file"}]
cleaned_df["metadata"] = raw_df[metadata_cols].to_dict(orient="records")

print("Cleaned shape:", cleaned_df.shape)
display(cleaned_df.head())
plot_count(cleaned_df["label"], "Cleaned NormBank label distribution")


In [ ]:
if SAVE_OUTPUTS:
    write_jsonl(OUT_DIR / "normbank.jsonl", cleaned_df.to_dict(orient="records"))
    write_csv(OUT_DIR / "normbank.csv", cleaned_df.to_dict(orient="records"))
    label_summary = [{"labels": cleaned_df["label"].value_counts().to_dict()}]
    write_jsonl(OUT_DIR / "label_summary.jsonl", label_summary)
    write_csv(OUT_DIR / "label_summary.csv", label_summary)
    print("Wrote cleaned NormBank outputs to", OUT_DIR)
else:
    print("Preview only. Set SAVE_OUTPUTS = True and rerun this cell to write cleaned files.")
